In [ ]:
import re

In [ ]:
text="hello iam student. I'am from Nepal"
results=re.split(r'([,.]| \s)',text)
print(results)

In [ ]:
results=[item for item in results if item.strip()] # removes white spaces

In [ ]:
preprocessed=re.split(r'([,.?;:\'"/-]|\s|--)',text)

In [ ]:
with open(r"story.text",encoding="utf-8") as f:
    raw_text=f.read()
len(raw_text)

In [ ]:
preprocessed=re.split(r'([,.?;:\'"/-]|\s|--)',raw_text)
preprocessed=[items for items in preprocessed if items.strip()]

In [ ]:
preprocessed[:4]
len(preprocessed)

In [ ]:
all_words=sorted(set(preprocessed))
vocab_len=len(all_words)
print(f"length of vocabulary is : {vocab_len}")

In [ ]:
all_words.extend(["<|endoftext|>","<|unk|>"])
print(f"length of new vocabulary is {len(all_words)}")

In [ ]:
vocab={token : integer for integer , token in enumerate(all_words)}

In [ ]:
class simpleTokenizerV1():
    def __init__(self,vocab):
        self.str_int=vocab
        self.int_str={integers : strings for strings , integers in vocab.items()}
    
    def encode(self,text):
        preprocessed=re.split(r'([,.?;:\'"/-]|\s|--)',text)
        preprocessed=[items for items in preprocessed if items.strip()]
        preprocessed=[items if items in self.str_int else "<|unk|>" for items in preprocessed]
        ids=[self.str_int[s] for s in preprocessed]
        return ids
    
    def decode(self,ids):
        text=" ".join([self.int_str[id] for id in ids])
        text=re.sub(r'\s+([,.?\'"-()])',r'\1',text)
        return text
        
        

In [ ]:
tokenizer=simpleTokenizerV1(vocab)
text="from everything to my him is anything. for this We used"
id=tokenizer.encode(text)


In [ ]:
id

In [ ]:
text=tokenizer.decode(id)

In [ ]:
text

In [ ]:
import tiktoken
import importlib

In [ ]:
bpt=tiktoken.get_encoding("gpt2")
text=("hello my name is lalit raman Mishra this things form every we are the since run rnuest for being we.  <|endoftext|>"
      "Everything is Fine")
integers=bpt.encode(text,allowed_special={"<|endoftext|>"})
print(integers)

In [ ]:
with open('story.text','r') as f:
    raw_text=f.read()

In [ ]:
enc_text=tokenizer.encode(raw_text)

In [ ]:
context_size=4


In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self,text,tokenizer,max_length,stride):
        self.input_ids=[]
        self.output_ids=[]
        # tokenize the entire text
        token_ids=tokenizer.encode(text)
        
        for i in range(0,len(token_ids)-max_length,stride):
            input_chunk=token_ids[i:i+max_length]
            output_chunk=token_ids[i+1: i+1+max_length]
            self.input_ids.append(torch.tensor(input_chunk))
            self.output_ids.append(torch.tensor(output_chunk))
            
    def __len__(self):
        return len(self.input_ids)
    
    def __getitem__(self, idx):
        return self.input_ids[idx],self.output_ids[idx]
        
            

In [ ]:
def create_dataloaderV1(text,batch_size=4,max_len=256,stride=128,shuffle=True,drop_last=True,num_workers=0):
    tokenizer=tiktoken.get_encoding("gpt2")
    dataset=GPTDatasetV1(text,tokenizer,max_len,stride)
    
    data_loader=DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
        
    )
    return data_loader


In [ ]:
data_loader=create_dataloaderV1(raw_text,1,4,1,False)
print(len(data_loader))

In [ ]:
data_iter=iter(data_loader)
first_batch=next(data_iter)
print(first_batch)

In [ ]:
first_batch=next(data_iter)
print(first_batch)

In [ ]:
import gensim.downloader as api

In [ ]:
model=api.load("word2vec-google-news-300")

In [ ]:
word_vectors=model
print(word_vectors["computer"])

In [ ]:
print(word_vectors.most_similar(positive=["king","Woman"],negative=["man"],topn=5))

In [ ]:
word_vectors.most_similar(
    positive=["king", "woman"],
    negative=["man"],
    topn=5
)

In [ ]:
input_ids= torch.tensor([2,3,5,1])
vocab_size=6
output_dim=3
torch.manual_seed(13)
embedding_layer=torch.nn.Embedding(vocab_size,output_dim)

In [ ]:
print(embedding_layer.weight)

In [ ]:
print(embedding_layer(torch.tensor([3])))

In [ ]:
print(embedding_layer(input_ids))

In [ ]:
vocab_size=50257
output_dimension=256
token_embedding_layers= torch.nn.Embedding(vocab_size,output_dimension)

In [ ]:
token_embedding_layers.weight.shape

In [ ]:
max_len=4
output_dim=256
data_loader=data_loader=create_dataloaderV1(raw_text,8,4,1,False)

In [ ]:
data_iter=iter(data_loader)
input,output=next(data_iter)
print(input)


In [ ]:
token_embedding=token_embedding_layers(input)
token_embedding.shape

In [ ]:
context_len=max_len
pos_embeding_layer=torch.nn.Embedding(context_len, output_dim)
pos_embeding_layer.weight.shape

In [ ]:
pos_embedding=pos_embeding_layer(torch.arange(context_len))


In [ ]:
pos_embedding.shape

In [ ]:
final_embedding=pos_embedding + token_embedding